# 1. Files conversion

In [1]:
from pathlib import Path
from PIL import Image

for file_path in Path('data/new_images').iterdir():
    if file_path.is_file() and file_path.suffix.lower() in ('.tif', '.tiff'):
        new_path = file_path.with_suffix('.png')
        try:
            with Image.open(file_path) as img:
                img.save(new_path, 'PNG')
        except Exception as e:
            print(f"Error converting {file_path.name}: {e}")

# 2. Mask Pregeneration - raw data

## 2.1. SegRefiner

https://github.com/MengyuWang826/SegRefiner/

Not a suitable solution. It's an academic model that is not suited for own datasets due to very complicated setup.

## 2.2. Modular U-Net for Glass Fibers


https://doi.org/10.5281/zenodo.4601560



In [ ]:
import os
import numpy as np
import tensorflow as tf
from PIL import Image

MODEL_PATH = "models/unet_pretrained/unet2d/paper-unet-2d/paper-unet-2d.full-f32.fold000.1612-356-814/paper-unet-2d.full-f32.fold000.1612-356-814.autosaved.200-0.013936.hdf5"
INPUT_DIR = "data/raw_images/"
OUTPUT_DIR = "generated_data/auto_masks_unet_for_fibers/"
TILE_SIZE = 160

def generate_masks_from_png(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR, model_path=MODEL_PATH):
    os.makedirs(output_dir, exist_ok=True)
    model = tf.keras.models.load_model(model_path, compile=False)
    
    image_files = sorted([f for f in os.listdir(input_dir) if f.lower().endswith('.png')])
    
    for filename in image_files:
        print(f"Processing {filename}...")
        filepath = os.path.join(input_dir, filename)
        img = Image.open(filepath).convert('L')
        img_np = np.array(img).astype(np.float32) / 255.0
        
        h, w = img_np.shape
        full_mask = np.zeros((h, w), dtype=np.uint8)

        for y in range(0, h, TILE_SIZE):
            for x in range(0, w, TILE_SIZE):
                tile = img_np[y:y+TILE_SIZE, x:x+TILE_SIZE]
                
                pad_h = TILE_SIZE - tile.shape[0]
                pad_w = TILE_SIZE - tile.shape[1]
                if pad_h > 0 or pad_w > 0:
                    tile = np.pad(tile, ((0, pad_h), (0, pad_w)), mode='constant')

                tile_input = np.expand_dims(np.expand_dims(tile, axis=-1), axis=0)
                
                prediction = model.predict(tile_input, verbose=0)
                
                if prediction.shape[-1] > 1:
                    tile_mask = np.argmax(prediction[0], axis=-1).astype(np.uint8)
                else:
                    tile_mask = (prediction[0] > 0.5).astype(np.uint8).squeeze(axis=-1)

                actual_h = TILE_SIZE - pad_h
                actual_w = TILE_SIZE - pad_w
                full_mask[y:y+actual_h, x:x+actual_w] = tile_mask[:actual_h, :actual_w]

        full_mask_output = (full_mask * 255).astype(np.uint8)
        Image.fromarray(full_mask_output).save(os.path.join(output_dir, filename))

generate_masks_from_png()

I0000 00:00:1777574204.542496   19389 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777574204.543017   19389 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777574204.830072   19389 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777574205.864815   19389 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

Processing 0027_00272_01.png...
Processing 0027_00272_02.png...
Processing 0027_00272_03.png...
Processing 0030_00302_01.png...
Processing 0030_00302_02.png...
Processing 0030_0302_01.png...
Processing 0030_0302_02.png...
Processing 0030_0302_03.png...
Processing 0030_0302_04.png...
Processing 0030_0302_05.png...
Processing 0030_0302_06.png...
Processing 0100_01008_01.png...
Processing 0100_01008_02.png...
Processing 0100_01008_03.png...
Processing 0100_01008_04.png...
Processing 0100_01008_05.png...
Processing 0100_01008_06.png...
Processing 0100_01008_07.png...
Processing 0100_01008_08.png...
Processing 0100_01008_09.png...
Processing 0100_01008_10.png...
Processing 0100_01008_11.png...
Processing 0100_01008_12.png...
Processing 0100_1008_01.png...
Processing 0100_1008_02.png...
Processing 0100_1008_03.png...
Processing 0100_1008_04.png...
Processing 0100_1008_05.png...
Processing 0100_1008_06.png...
Processing 0100_1008_07.png...
Processing 0100_1008_08.png...
Processing 0100_1008_0